# Default Orchestrator Demo (CVAE + Chemist Agent V1 + ESM Global L2 Biologist)

In [1]:
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"Torch device availability: cuda={torch.cuda.is_available()}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    torch.random.manual_seed(SEED)
except Exception:
    pass

Repo root: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline
Torch device availability: cuda=True


In [2]:
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader

DATA_PATH = REPO_ROOT / "database" / "training_data.json"

loader = JSONDataLoader()
loader.load_data(
    source=str(DATA_PATH),
    columns=[
        "sequence",
        "length",
        "ph",
        "molecular_weight",
        "logp",
        "net_charge",
        "isoelectric_point",
        "hydrophobicity",
        "cathionicity",
    ],
    fillna_defaults={
        "length": 10,
        "ph": 7.0,
        "molecular_weight": 1500.0,
        "logp": 0.0,
        "net_charge": 0.0,
        "isoelectric_point": 7.0,
        "hydrophobicity": 0.0,
        "cathionicity": 0.0,
    },
    normalize_sequence=True,
    sequence_column="sequence",
    keep_standard_amino_acids_only=True,
    length_column="length",
    length_range=(1, 14),
)

df = loader.get_data().copy()

print(f"Loaded records after dataloader preprocessing: {len(df)}")
print("NaN count per column:")
print(df.isna().sum())
display(df.head(3))

Loaded records after dataloader preprocessing: 4410
NaN count per column:
sequence             0
length               0
ph                   0
molecular_weight     0
logp                 0
net_charge           0
isoelectric_point    0
hydrophobicity       0
cathionicity         0
dtype: int64


,sequence,length,ph,molecular_weight,logp,net_charge,isoelectric_point,hydrophobicity,cathionicity
0,KVVVKWVVKVVK,12,7.0,1648.291,5.6026,5.0,14.0,-1.07,4
1,LFIFFF,6,7.0,832.059,3.2860,1.0,14.0,-3.25,0
2,KAAAKWAAKAAK,12,7.0,1451.913,1.1499,5.0,14.0,0.33,4


In [3]:
from peptide_pipeline.generator.cvae_generator_agent.cvae_generator import CVAEGenerator

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA)}
PAD_IDX = 20
VOCAB_SIZE = 21
MAX_LEN = 14


def encode_one_hot_with_pad(sequences, max_len=MAX_LEN):
    x = torch.zeros(len(sequences), max_len * VOCAB_SIZE, dtype=torch.float32)
    for i, seq in enumerate(sequences):
        for pos in range(max_len):
            x[i, pos * VOCAB_SIZE + PAD_IDX] = 1.0
        for pos, aa in enumerate(seq[:max_len]):
            if aa in AA_TO_IDX:
                x[i, pos * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, pos * VOCAB_SIZE + AA_TO_IDX[aa]] = 1.0
    return x


def build_condition_tensor(dataframe, model):
    # Datalaoder already normalizes and fills defaults. We only map baseline fields
    # into the condition layout expected by CVAEGenerator.
    required_cols = [
        "length",
        "molecular_weight",
        "net_charge",
        "isoelectric_point",
        "hydrophobicity",
        "cathionicity",
        "logp",
    ]
    missing = [c for c in required_cols if c not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns for conditioning: {missing}")

    c_df = dataframe[required_cols].copy()
    if c_df.isna().any().any():
        raise ValueError("NaN values detected after dataloader preprocessing.")

    cond = torch.zeros(len(c_df), model.condition_dim, dtype=torch.float32)

    # CVAE property order:
    # [size, molecular_weight, net_charge_pH5_5, isoelectric_point, hydrophobicity,
    #  cathionicity, hydrophobic_moment, logp, boman_index, h_bond_donors,
    #  h_bond_acceptors, tpsa]
    cond[:, 0] = torch.tensor(c_df["length"].values, dtype=torch.float32)
    cond[:, 1] = torch.tensor(c_df["molecular_weight"].values, dtype=torch.float32)
    cond[:, 2] = torch.tensor(c_df["net_charge"].values, dtype=torch.float32)
    cond[:, 3] = torch.tensor(c_df["isoelectric_point"].values, dtype=torch.float32)
    cond[:, 4] = torch.tensor(c_df["hydrophobicity"].values, dtype=torch.float32)
    cond[:, 5] = torch.tensor(c_df["cathionicity"].values, dtype=torch.float32)
    cond[:, 6] = 0.5
    cond[:, 7] = torch.tensor(c_df["logp"].values, dtype=torch.float32)
    cond[:, 8] = 0.0
    cond[:, 9] = 5.0
    cond[:, 10] = 5.0
    cond[:, 11] = 100.0

    return cond


sequences = df["sequence"].tolist()
lengths = torch.tensor(df["length"].astype(int).values, dtype=torch.long)

cvae = CVAEGenerator(max_len=MAX_LEN, latent_dim=64, hidden_dim=256, condition_dim=32)

x = encode_one_hot_with_pad(sequences, max_len=MAX_LEN)
conditions = build_condition_tensor(df, cvae)

# Move tensors to model device
x = x.to(cvae.device)
conditions = conditions.to(cvae.device)
lengths = lengths.to(cvae.device)

print(f"x shape: {tuple(x.shape)}")
print(f"conditions shape: {tuple(conditions.shape)}")
print(f"lengths shape: {tuple(lengths.shape)}")
print(f"CVAE device: {cvae.device}")
print("Condition columns loaded from baseline schema: length, ph, molecular_weight, logp, net_charge, isoelectric_point, hydrophobicity, cathionicity")

x shape: (4410, 294)
conditions shape: (4410, 32)
lengths shape: (4410,)
CVAE device: cuda
Condition columns loaded from baseline schema: length, ph, molecular_weight, logp, net_charge, isoelectric_point, hydrophobicity, cathionicity


In [4]:
CVAE_OUT = REPO_ROOT / "experiments" / "cvae_full.pth"

if CVAE_OUT.exists():
    try:
        cvae.load_model(str(CVAE_OUT))
    except AttributeError:
        state = torch.load(CVAE_OUT, map_location=cvae.device)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        cvae.load_state_dict(state)
        cvae.to(cvae.device)
    print(f"Loaded existing CVAE from: {CVAE_OUT}")
else:
    # Train CVAE on training_data.json via the dataloader output
    cvae.train_model(
        data=x,
        conditions=conditions,
        lengths=lengths,
        epochs=200,
        batch_size=64,
        lr=1e-3,
        kl_anneal_epochs=60,
    )
    cvae.save_model(str(CVAE_OUT))
    print(f"Saved trained CVAE to: {CVAE_OUT}")

Loaded existing CVAE from: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/experiments/cvae_full.pth


In [5]:
from peptide_pipeline.chemist.chemist_agent.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.chemist_agent.chemist_agent import ChemistAgent
from peptide_pipeline.biologist.esm_l2_bio_agent.esm_biologist_global_l2 import ESMBiologistGlobalL2
from peptide_pipeline.orchestrator.orchestrator_agent.orchestrator import Orchestrator

# Target profile from DBAASPS_373
FINAL_TARGET = {
    "dbaasp_id": "DBAASPS_373",
    "sequence": "KLFKRWKHLFR",
    "length": 11,
    "smiles": "CC(C)C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)O",
    "ph": None,
    "molecular_weight": 1558.9480000000003,
    "logp": -0.992100000000006,
    "net_charge": 5.0,
    "isoelectric_point": 12.18,
    "hydrophobicity": 1.05,
    "cathionicity": 6,
}

# ChemistConfig expects a numeric pH; default to 7.0 when target pH is missing.
chemist_ph = 7.0 if FINAL_TARGET["ph"] is None else float(FINAL_TARGET["ph"])

# Build chemist constraints around target values.
# Note: cathionicity is intentionally not constrained because the installed
# peptides package here does not expose Peptide.cathionicity().
chemist_config = ChemistConfig(
    ph=chemist_ph,
    length=RangeTarget(min=8.0, max=14.0, target=float(FINAL_TARGET["length"]), weight=1.0),
    molecular_weight=RangeTarget(min=1200.0, max=2000.0, target=float(FINAL_TARGET["molecular_weight"]), weight=1.0),
    logp=RangeTarget(min=-3.0, max=3.0, target=float(FINAL_TARGET["logp"]), weight=1.0),
    net_charge=RangeTarget(min=2.0, max=8.0, target=float(FINAL_TARGET["net_charge"]), weight=1.0),
    isoelectric_point=RangeTarget(min=9.0, max=14.0, target=float(FINAL_TARGET["isoelectric_point"]), weight=1.0),
    hydrophobicity=RangeTarget(min=-2.0, max=3.0, target=float(FINAL_TARGET["hydrophobicity"]), weight=1.0),
)

reference_peptide = FINAL_TARGET["sequence"]
print(f"Target ID: {FINAL_TARGET['dbaasp_id']}")
print(f"Reference peptide for ESM global L2 agent: {reference_peptide}")

chemist = ChemistAgent(config=chemist_config)
biologist = ESMBiologistGlobalL2(reference_peptide=reference_peptide, batch_size=16, score_temperature=50.0)

generator_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cvae = cvae.to(generator_device)
cvae.device = generator_device
print(f"Generator device: {cvae.device}")

orchestrator = Orchestrator(generator=cvae, chemist=chemist, biologist=biologist)

Target ID: DBAASPS_373
Reference peptide for ESM global L2 agent: KLFKRWKHLFR


Loading weights: 100%|██████████| 209/209 [00:00<00:00, 938.70it/s, Materializing param=encoder.layer.11.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Generator device: cuda


In [6]:
# Run default orchestrator loop
results = orchestrator.run(
    nb_iterations=6,
    nb_peptides=64,
    top_k=10,
    exploration_rate=0.15,
    initial_peptide=reference_peptide,
    final_target=FINAL_TARGET
)

orchestrator_df = pd.DataFrame(results)
if not orchestrator_df.empty:
    display(orchestrator_df)
else:
    print("No orchestrator results were returned.")

orchestrator_peptides = orchestrator_df["peptide"].tolist() if "peptide" in orchestrator_df.columns else []
print("Top orchestrator peptides:")
for i, pep in enumerate(orchestrator_peptides, start=1):
    print(f"{i:02d}. {pep}")

,peptide,score,combined_score,chemist_score,biologist_score,in_limits,properties,iteration
0,RKLFFKLFRRY,0.921352,0.921352,0.918812,0.923891,True,"{'length': 11, 'molecular_weight': 1573.950539...",1
1,WLFRRHFKLKK,0.897019,0.897019,0.877364,0.916674,True,"{'length': 11, 'molecular_weight': 1558.93884,...",1
2,RKLFKKLIRMW,0.893020,0.893020,0.865728,0.920313,True,"{'length': 11, 'molecular_weight': 1518.973139...",1
3,LLIKRFKRLLK,0.892966,0.892966,0.872679,0.913252,True,"{'length': 11, 'molecular_weight': 1427.886139...",6
4,KRLFRKIKWFL,0.892673,0.892673,0.859193,0.926154,True,"{'length': 11, 'molecular_weight': 1534.957139...",1
5,KLFKRKLRVII,0.892226,0.892226,0.887586,0.896866,True,"{'length': 11, 'molecular_weight': 1413.859339...",6
6,KLILRKVRKLI,0.892216,0.892216,0.890512,0.893919,True,"{'length': 11, 'molecular_weight': 1379.84214,...",6
7,RKIKKLLKHLI,0.889799,0.889799,0.865795,0.913804,True,"{'length': 11, 'molecular_weight': 1389.83724,...",4
8,KKLVRRIIKIL,0.889366,0.889366,0.893309,0.885422,True,"{'length': 11, 'molecular_weight': 1379.84214,...",5
9,KFRFFHKKLFK,0.889322,0.889322,0.862492,0.916153,True,"{'length': 11, 'molecular_weight': 1525.906039...",1


Top orchestrator peptides:
01. RKLFFKLFRRY
02. WLFRRHFKLKK
03. RKLFKKLIRMW
04. LLIKRFKRLLK
05. KRLFRKIKWFL
06. KLFKRKLRVII
07. KLILRKVRKLI
08. RKIKKLLKHLI
09. KKLVRRIIKIL
10. KFRFFHKKLFK


In [7]:
import gc
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()